# 📱 Kaggle Playground Series S6E8: Smartphone Addiction Prediction
### **End-to-End Competitive Machine Learning, Stacking & Diagnostic Workflow**
**Author:** Jegadeesh D  
**Metric:** Area Under the ROC Curve (ROC AUC) & Binary Classification Accuracy  
**Target:** `addicted_label` (Binary Classification: $0 = \text{Not Addicted}, 1 = \text{Addicted}$)

---

## 📌 Step 1: Problem Definition & Competition Framework

### **Objective**
Predict whether an individual exhibits smartphone addiction based on behavioral usage patterns, screen time distributions, sleep metrics, and demographic indicators.

### **Evaluation Metric**
Primary evaluation metric is **Area Under the ROC Curve (ROC AUC)**:
$$\text{ROC AUC} = \int_0^1 \text{TPR}(\text{FPR}^{-1}(t)) \, dt$$

### **Dataset Properties**
- Training samples: 691,369 rows $\times$ 14 columns
- Test samples: 296,302 rows $\times$ 13 columns
- Target prior: ~70.9% positive class (`addicted_label = 1`)

In [ ]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata, ks_2samp
from scipy.special import expit, logit

# ML Frameworks
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, classification_report
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print('Environment and ML frameworks initialized successfully!')

## 📥 Step 2: Data Collection & Schema Validation

In [ ]:
data_dir = '../data'
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f'Train Shape: {train.shape[0]:,} rows x {train.shape[1]} columns')
print(f'Test Shape:  {test.shape[0]:,} rows x {test.shape[1]} columns')
display(train.head(5))

## 🧹 Step 3: Missingness Mechanisms & Distribution Stability

In [ ]:
missing_df = pd.DataFrame({
    'Train_Nulls': train.isnull().sum(),
    'Train_Null_%': (train.isnull().mean() * 100).round(2),
    'Test_Nulls': test.isnull().sum(),
    'Test_Null_%': (test.isnull().mean() * 100).round(2)
})
display(missing_df.sort_values(by='Train_Null_%', ascending=False))

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 
            'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']

print('--- Two-Sample Kolmogorov-Smirnov Test (Train vs Test) ---')
for col in num_cols:
    stat, pval = ks_2samp(train[col].dropna(), test[col].dropna())
    print(f'{col:25s} | KS-Stat: {stat:.5f} | p-value: {pval:.4f}')

## ⚙️ Step 4: Advanced 74-Feature Engineering Pipeline

Features include:
1. **Explicit Missingness Indicators (`is_missing_*`)** + unobserved sensor count `num_missing`.
2. **Safe filled arithmetic** ensuring no cascading missing values in domain formulas.
3. **Domain Time-Budgets**: `total_accounted_hours`, `unaccounted_hours`, `waking_hours`, `screen_fraction_of_waking`.
4. **Volatility & Fragmentation**: `weekend_vs_weekday_diff`, `weighted_weekly_screen`, `avg_unlock_minutes`, `interaction_density`.
5. **Multi-Cohort GroupBy Aggregations & Z-Scores**: Peer-relative statistics for `(Age, Gender)`, `(Stress, Impact)`, and `(Gender, Stress)`.
6. **Cross-Product Interactions**: `screen_x_social`, `screen_x_weekend`, `screen_x_opens`.

In [ ]:
import sys
sys.path.append('../src')
from features import create_features

train_fe, test_fe, feature_cols = create_features(train, test)
print(f'Total Engineered Features for Modeling ({len(feature_cols)}):\n', feature_cols)
display(train_fe[feature_cols].head())

## 🎯 Step 5: Multi-Model 5-Fold Stratified Cross-Validation

We train 4 heterogeneous architectures:
1. **Deep LightGBM** (`num_leaves=191`, `max_depth=10`, `learning_rate=0.02`)
2. **Regularized Hist XGBoost** (`max_depth=8`, `min_child_weight=30`)
3. **Symmetric CatBoost** (`depth=7`, `l2_leaf_reg=4.0`)
4. **PyTorch Tabular ResNet** (Entity Embeddings + Residual Blocks with BatchNorm & SiLU)

In [ ]:
from train import train_pipeline
train_pipeline(n_splits=5, random_state=42, include_nn=True)

## 🧪 Step 6: Multi-Model Continuous Optimization & Meta-Learner Stacking

In [ ]:
from ensemble import optimize_and_submit
optimize_and_submit()

## 📊 Step 7: ROC Diagnostics & Decision Threshold Calibration

In [ ]:
models_dir = '../models'
data = np.load(os.path.join(models_dir, 'oof_predictions.npz'))
lgb_oof = data['lgb_oof']
xgb_oof = data['xgb_oof']
cat_oof = data['cat_oof']
y_true = data['y_true']

fpr_lgb, tpr_lgb, _ = roc_curve(y_true, lgb_oof)
fpr_xgb, tpr_xgb, _ = roc_curve(y_true, xgb_oof)

plt.figure(figsize=(9, 6))
plt.plot(fpr_lgb, tpr_lgb, label=f'LightGBM (OOF AUC = {roc_auc_score(y_true, lgb_oof):.5f})', color='#2563EB', lw=2)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (OOF AUC = {roc_auc_score(y_true, xgb_oof):.5f})', color='#DC2626', lw=2, linestyle='--')
plt.plot([0, 1], [0, 1], color='gray', linestyle=':')
plt.title('Out-of-Fold Receiver Operating Characteristic (ROC) Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

## 📦 Step 8: Final Submission Verification

In [ ]:
sub_path = '../data/submission_optimized_ensemble.csv'
submission = pd.read_csv(sub_path)
print(f'Loaded Submission File: {sub_path}')
print(f'Total Rows: {len(submission):,} | Columns: {list(submission.columns)}')
print(f'Null count: {submission.isna().sum().sum()}')
print(f'Mean Addiction Probability: {submission["addicted_label"].mean():.5f}')

assert submission.shape == (296302, 2)
assert submission['addicted_label'].isna().sum() == 0
print('All verification checks passed successfully!')
display(submission.head(10))